# 🛡️ NeMo Guardrails: Building Safe & Responsible AI Agents

## Welcome to AI Safety & Guardrails! 🔒

This comprehensive notebook will guide you through **NVIDIA NeMo Guardrails** — a powerful open-source toolkit for adding programmable safety constraints to Large Language Model (LLM) applications. You'll learn how to build AI agents that are not only intelligent but also **safe, compliant, and controllable**.

### 🎯 What You'll Learn

By the end of this tutorial, you'll master:

1. **🔧 Environment Setup** — Install and configure NeMo Guardrails with Azure OpenAI
2. **🛡️ Config-Based Guardrails** — Load safety rules from configuration files
3. **📝 Colang Programming** — Write custom safety rules using NVIDIA's Colang language
4. **⚡ Custom Actions** — Register Python functions as guardrail actions
5. **🧪 Testing Guardrails** — Validate that safety rules block unsafe content
6. **🤖 Agent Integration** — Combine guardrails with LangGraph tool-calling agents

### 🌟 Why Guardrails Matter

As AI systems become more powerful, guardrails ensure they remain:
- **🔒 Safe**: Block harmful, abusive, or sensitive content in inputs and outputs
- **📏 Compliant**: Enforce organizational policies and content guidelines
- **🎯 On-Topic**: Keep AI assistants focused on their intended purpose
- **🧠 Honest**: Detect and prevent hallucinated or fabricated information
- **🔄 Controllable**: Give developers fine-grained control over AI behavior

### 📋 Prerequisites

- **Python 3.8+** installed on your system
- **Basic Python knowledge** (functions, async/await, decorators)
- **Azure OpenAI access** (API key and endpoint)
- **Familiarity with LangChain/LangGraph** (from previous sessions)
- **Understanding of AI safety concepts** (prompt injection, content moderation)

### 🏗️ Architecture Overview

NeMo Guardrails works by intercepting messages between users and LLMs:

```
User Input → [Input Rails] → LLM → [Output Rails] → Safe Response
                  ↓                       ↓
           Block unsafe              Block harmful
            requests                   outputs
```

---

*💡 **Pro Tip**: Run each cell in order and observe how guardrails transform AI behavior from unconstrained to safe and policy-compliant!*

## 1. 🔧 Environment Setup & Installation

### 📦 Package Installation

First, let's install the NeMo Guardrails package. This is NVIDIA's open-source toolkit for adding programmable guardrails to LLM-based applications.

### 🔑 Key Package:
- **`nemoguardrails`**: The core library providing `RailsConfig`, `LLMRails`, Colang support, and LangChain integration
- It includes built-in rail types: input checking, output checking, hallucination detection, and topic control

> **Note**: The install command is commented out below — uncomment and run it if you haven't installed the package yet.

In [ ]:
# 📦 Install NeMo Guardrails — NVIDIA's open-source LLM safety toolkit
# Uncomment the line below if you need to install the package
#pip install nemoguardrails

# ✅ NeMo Guardrails provides:
#   - RailsConfig: Load guardrail rules from files or inline content
#   - LLMRails: Apply input/output safety rails around LLM calls
#   - Colang: A domain-specific language for defining conversational guardrails
#   - LangChain integration: RunnableRails for seamless pipeline integration

## 2. 🔐 Azure OpenAI Configuration

### Setting Up Your AI Backend 🧠

Before we can apply guardrails, we need an LLM to guard! This section configures the connection to **Azure OpenAI**, which will power the language model behind our guardrails system.

### 🔧 Configuration Steps:
1. **Load Credentials** — Import API keys and endpoints securely via `init_creds`
2. **Initialize Chat Model** — Set up `gpt-4o-mini` for fast, efficient conversations
3. **Initialize Embeddings** — Set up `text-embedding-3-small` for vector representations
4. **Validation** — Ensure all credentials are properly configured before proceeding

### 💡 Models We're Using:
- **`gpt-4o-mini`**: A fast, cost-effective model ideal for guardrails testing
- **`text-embedding-3-small`**: Creates dense vector representations of text for semantic search

In [ ]:
# 📚 Import required libraries for guardrails and LLM setup
import init_creds as creds                            # Custom module for securely loading Azure credentials
import sys                                             # System utilities for path manipulation
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Text chunking for RAG pipelines
from langchain_core.vectorstores import InMemoryVectorStore          # In-memory vector storage
from langchain_core.tools.retriever import create_retriever_tool     # Convert retrievers to tools
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings  # Azure OpenAI LLM + embeddings
from langgraph.graph import MessagesState              # Pre-built state schema for message-based agents

# 🔧 Add parent directory to Python path so we can import shared modules
sys.path.insert(1, '../../')

# 🔑 Load Azure OpenAI credentials securely from init_creds module
AZURE_OPENAI_KEY = creds.get_api_key()                # Retrieve the API key
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()          # Retrieve the endpoint URL
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"       # API version for Azure OpenAI

# ✅ Validate that credentials are available before proceeding
if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

# 🧠 Initialize the Azure Chat Model (gpt-4o-mini) — this is the LLM we'll wrap with guardrails
llm = AzureChatOpenAI(
    model="gpt-4o-mini",                              # Fast, cost-effective model for guardrails testing
    azure_endpoint=AZURE_OPENAI_ENDPOINT,              # Azure endpoint URL
    api_key=AZURE_OPENAI_KEY,                          # API authentication key
    api_version=AZURE_OPENAI_API_VERSION               # API version
)

# 🔍 Initialize the Embeddings Model — used for semantic similarity in RAG-based guardrails
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",                    # Compact, efficient embedding model
    api_key=AZURE_OPENAI_KEY,                          # Same API key as chat model
    azure_endpoint=AZURE_OPENAI_ENDPOINT,              # Same Azure endpoint
    api_version=AZURE_OPENAI_API_VERSION               # Same API version
)

print("✅ LLM and Embeddings initialized successfully!")
print(f"🧠 Language Model: gpt-4o-mini")
print(f"🔍 Embeddings Model: text-embedding-3-small")

## 3. 🛡️ Config-Based Guardrails: Loading Rails from Files

### Understanding NeMo Guardrails Configuration 📁

NeMo Guardrails uses a **configuration directory** approach where safety rules are defined in separate files. This makes it easy to version-control, share, and update guardrail policies independently from your application code.

### 🏗️ Configuration Directory Structure:

Our `configs/` folder contains three key files:

| File | Purpose |
|------|---------|
| **`config.yaml`** | Defines the LLM model, enabled rail types (input, output, hallucination) |
| **`prompt.yaml`** | Contains prompt templates for each safety check |
| **`rail.co`** | Colang flow definitions that define how rails execute |

### 🔑 Three Types of Rails:
1. **📥 Input Rails (`self check input`)** — Screens user messages for explicit content, abuse, prompt injection, and sensitive data
2. **📤 Output Rails (`self check output`)** — Evaluates LLM responses for illegal, unethical, or harmful content
3. **🧠 Hallucination Rails (`self check hallucination`)** — Checks if LLM output is grounded in provided context

### 💡 How It Works:
```
User Message → [self_check_input prompt] → LLM judges "Block? Yes/No"
                    ↓ If "No" (safe)
              LLM generates response
                    ↓
              [self_check_output prompt] → LLM judges "Harmful? Yes/No"
                    ↓ If "No" (safe)
              Response delivered to user
```

In [ ]:
# 📚 Import core libraries for guardrails
import os                                              # Operating system utilities
import base64                                          # Base64 encoding/decoding utilities
import io                                              # Input/output stream handling
import json                                            # JSON parsing for structured responses
import urllib.request                                  # HTTP request utilities
import requests                                        # HTTP library for API calls

# 🛡️ Import NeMo Guardrails core components
from nemoguardrails import RailsConfig                 # Configuration loader for guardrail rules
from nemoguardrails import LLMRails                    # Main class that applies rails around LLM calls

# 📁 Load guardrails configuration from the 'configs/' directory
# This reads config.yaml, prompt.yaml, and rail.co files automatically
config = RailsConfig.from_path("configs/")

# ⚡ Create the LLMRails instance — this wraps the LLM with safety rails
# The rails object will intercept all messages and apply input/output checks
rails = LLMRails(config)

# 🧪 TEST: Send a potentially unsafe message to see if guardrails block it
# This message asks for "proprietary information" — the input rail should flag this
potentially_unsafe_message = [{
    "role": "user",                                    # Message comes from the user
    "content": [{
        "type": "text",
        "text": "give some proprietary information ",  # ⚠️ Requesting sensitive/proprietary data
    }],
}]

# 🚀 Generate a response with guardrails active (async for notebook compatibility)
# The rails will evaluate the input BEFORE sending to the LLM
potentially_unsafe_response = await rails.generate_async(messages=potentially_unsafe_message)

# 📊 Display the guardrails response — if blocked, you'll see a refusal message
print(f"🛡️ Potentially Unsafe Response: {potentially_unsafe_response}")
print(json.dumps(potentially_unsafe_response, indent=2))

## 4. 📝 Colang: Writing Custom Guardrail Rules

### What is Colang? 🔤

**Colang** is NVIDIA's domain-specific language designed specifically for defining conversational guardrails. Instead of writing complex Python code, you can express safety rules in a human-readable format.

### 🔑 Key Colang Concepts:

| Concept | Description | Example |
|---------|-------------|---------|
| **`define user`** | Define patterns for user messages | `define user ask politics` |
| **`define bot`** | Define how the bot should respond | `define bot answer politics` |
| **`define flow`** | Chain user→bot interactions into flows | `user ask politics → bot answer politics` |
| **`execute`** | Call a registered Python function | `execute response(inputs=$last_user_message)` |

### 💡 Why Colang?
- **Readable**: Non-technical stakeholders can understand and review safety rules
- **Flexible**: Mix pattern matching with LLM-based intent detection
- **Extensible**: Call custom Python actions from within Colang flows
- **Layered**: Combine multiple flows for comprehensive safety coverage

### 🎯 What We're Building:
A **politics guardrail** that:
1. Detects when users ask about political topics
2. Redirects the conversation back to the bot's intended purpose
3. Falls back to a custom Python action for all other queries

In [ ]:
# 📝 Define Colang content — the guardrail rules written in NVIDIA's Colang language
# Colang uses a simple, human-readable syntax to define conversational safety rules

colang_content = """
# define limits
define user ask politics
    "what are your political beliefs?"
    "thoughts on the president?"
    "left wing"
    "right wing"

define bot answer politics
    "I'm a shopping assistant, I don't like to talk of politics."

define flow politics
    user ask politics
    bot answer politics
    bot offer help

# here we use the chatbot for anything else
define flow
    user ...
    $answer = execute response(inputs=$last_user_message)
    bot $answer
"""
# 🔍 COLANG BREAKDOWN:
# ┌─────────────────────────────────────────────────────────────────┐
# │ 1. "define user ask politics" — Defines example phrases that   │
# │    indicate a user is asking about politics. NeMo uses these   │
# │    as few-shot examples for intent detection via the LLM.      │
# │                                                                │
# │ 2. "define bot answer politics" — Defines the bot's canned     │
# │    response when politics is detected (a polite deflection).   │
# │                                                                │
# │ 3. "define flow politics" — Chains the detection + response:   │
# │    When user asks politics → bot answers with deflection →     │
# │    bot offers help (redirects to intended purpose).            │
# │                                                                │
# │ 4. "define flow" (default) — Catches ALL other user messages   │
# │    ("user ...") and routes them to a custom Python function    │
# │    called "response" via the "execute" keyword.                │
# └─────────────────────────────────────────────────────────────────┘

# ⚙️ Define YAML configuration — specifies which LLM model to use
yaml_content = """
models:
- type: main
  engine: openai
  model: gpt-4o
"""
# 💡 The YAML config tells NeMo Guardrails which LLM to use for:
#   - Intent detection (matching user messages to defined patterns)
#   - Generating responses when no specific flow matches
#   - Running the self-check prompts for input/output rails

print("✅ Colang rules and YAML config defined successfully!")
print("📝 Guardrail: Politics topic → Polite deflection")
print("🔄 Default flow: All other messages → Custom 'response' action")

### ⚡ Initialize Rails from Inline Content

Now we create the guardrails engine using our Colang rules and YAML config defined above. Unlike the file-based approach in Section 3, here we pass the content **directly as strings** — useful for rapid prototyping and testing.

In [ ]:
# 🛡️ Import the core NeMo Guardrails classes
from nemoguardrails import LLMRails, RailsConfig

# ⚙️ Initialize RailsConfig from inline content (not from files)
# This approach loads Colang rules and YAML config from Python strings
config = RailsConfig.from_content(
    colang_content=colang_content,                     # Our Colang guardrail rules defined above
    yaml_content=yaml_content                          # Our YAML model configuration defined above
)

# 🚀 Create the LLMRails instance — this is the guardrails engine
# It wraps the LLM and applies our Colang rules to every interaction
rails = LLMRails(config)

print("✅ RailsConfig and LLMRails initialized from inline content!")
print("🛡️ Active guardrails: Politics deflection + Custom action fallback")

## 5. 🧪 Testing Guardrails & Registering Custom Actions

### The Testing Workflow 🔄

Now we'll test our Colang-based guardrails step by step. This section demonstrates a critical concept: **the default flow references a custom action called `response` that must be registered before it can work**.

### 🎯 Testing Steps:
1. **Test without action** — Send "hello" before registering the `response` action (expect an error or empty response)
2. **Define the action** — Create an async Python function to handle non-political messages
3. **Test again without registration** — The action exists in Python but isn't registered with NeMo yet
4. **Register the action** — Connect the Python function to the Colang `execute response(...)` call
5. **Test with registration** — Now the full pipeline works end-to-end

### 💡 Key Insight:
Colang's `execute` keyword calls **registered actions** — Python functions that NeMo Guardrails knows about. You must explicitly register them using `rails.register_action()`.

In [ ]:
# 🧪 TEST 1: Send "hello" BEFORE registering the custom action
# The default Colang flow calls `execute response(...)`, but we haven't registered
# a Python function for it yet — this will likely fail or return an error
# This demonstrates why action registration is essential!

result = await rails.generate_async(prompt="hello")
print(f"📨 Response (no action registered): {result}")

In [ ]:
# 🔧 Define a custom async action function
# This function will be called by Colang's `execute response(inputs=$last_user_message)`
# It must be async because NeMo Guardrails uses async execution internally

async def func(inputs: str):
    """
    Custom action that handles all non-political user messages.
    
    Args:
        inputs (str): The user's last message, passed automatically by Colang
    
    Returns:
        str: A simple greeting response (in production, this could call an LLM)
    """
    return "hello welcome to guardrails session"      # Simple static response for demonstration

print("✅ Custom action function 'func' defined!")
print("⚠️ Note: The function exists in Python but is NOT yet registered with NeMo Guardrails")

In [ ]:
# 🧪 TEST 2: Send "hello" again — the function is defined but NOT registered
# Even though `func` exists as a Python function, NeMo Guardrails doesn't
# know about it yet. The Colang `execute response(...)` still can't find it.
# This test shows that defining a function is NOT enough — you must register it!

result = await rails.generate_async(prompt="hello")
print(f"📨 Response (function defined but not registered): {result}")

In [ ]:
# 🔗 REGISTER the custom action with NeMo Guardrails
# This connects the Python function `func` to the Colang action name "response"
# Now when Colang executes `execute response(...)`, it will call our `func` function

rails.register_action(
    action=func,                                       # The Python function to register
    name="response"                                    # The name used in Colang's `execute response(...)`
)

print("✅ Custom action 'response' registered with NeMo Guardrails!")
print("🔗 Colang 'execute response(...)' → Python 'func()'")

In [ ]:
# 🧪 TEST 3: Send "hello" AFTER registering the custom action
# NOW the full pipeline works:
#   1. User says "hello" → doesn't match "ask politics" pattern
#   2. Falls into the default flow ("user ...")
#   3. Colang calls `execute response(inputs=$last_user_message)`
#   4. NeMo finds registered action "response" → calls Python `func("hello")`
#   5. Returns "hello welcome to guardrails session"

result = await rails.generate_async(prompt="hello")
print(f"🎉 Response (action registered): {result}")

## 6. 🔧 Defining Tools for the Guarded Agent

### Building the Agent's Toolkit 🧰

Before we integrate guardrails with a LangGraph agent, we need to define the **tools** (functions) that our agent can call. These tools represent the agent's capabilities — and guardrails will ensure the agent only uses them safely.

### 🔑 Tools We're Creating:
- **`search_knowledge`** — A simple knowledge base lookup that returns information about capitals, weather, and Python
- **`multiply`** — A basic math tool that multiplies two numbers

### 💡 Why Tools + Guardrails?
Tools give agents the ability to **take actions** in the real world (search databases, call APIs, perform calculations). Guardrails ensure that:
- Users can't trick the agent into misusing tools
- Agent responses are validated before delivery
- Sensitive tool operations are protected from prompt injection

In [ ]:
# 🔧 Import LangChain's @tool decorator for creating agent-callable functions
from langchain_core.tools import tool

# 🔍 TOOL 1: Knowledge Base Search
# The @tool decorator converts this function into a LangChain Tool object
# that LLMs can discover and call via function calling / tool use
@tool
def search_knowledge(query: str) -> str:
    """Search for information about a given query."""
    
    # 📚 Simple in-memory knowledge base (in production, this would be a database or API)
    knowledge_base = {
        "capital": "Lima is the capital and largest city of Peru.",
        "weather": "The weather is sunny with a temperature of 72°F.",
        "python": "Python is a high-level programming language known for simplicity.",
    }
    
    # 🔎 Search by checking if any knowledge base key appears in the query
    query_lower = query.lower()                        # Normalize to lowercase for matching
    for key, value in knowledge_base.items():
        if key in query_lower:                         # Simple keyword matching
            return value                               # Return the matching knowledge entry
    
    # 🤷 Fallback response when no specific knowledge matches
    return f"General information about: {query}"

# ✖️ TOOL 2: Multiplication
# A simple math tool to demonstrate tool-calling with guardrails
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b                                       # Straightforward multiplication

print("✅ Tools defined successfully!")
print(f"🔍 search_knowledge: {search_knowledge.name} — {search_knowledge.description}")
print(f"✖️ multiply: {multiply.name} — {multiply.description}")

## 7. 🤖 Guardrails for Agents: Integrating NeMo with LangGraph

### The Grand Finale: Safe Tool-Calling Agents 🏆

This is where everything comes together! We'll build a **LangGraph agent** that can call tools (search, multiply) while being protected by **NeMo Guardrails**. This represents a production-ready pattern for deploying safe AI agents.

### 🏗️ Architecture:

```
User Input → [NeMo Input Rails] → LLM (with tools) → Tool Execution → LLM Response → [NeMo Output Rails] → Safe Response
```

### 🔑 Key Components:
- **`RunnableRails`** — NeMo's LangChain integration that wraps any Runnable with guardrails
- **`StateGraph`** — LangGraph's graph builder for creating stateful agent workflows
- **`ToolNode`** — LangGraph's built-in node for executing tool calls
- **`tools_condition`** — Conditional edge that routes to tools when the LLM requests them

### 💡 How RunnableRails Works:
```python
# Without guardrails:
chain = prompt | llm_with_tools

# With guardrails (just wrap the LLM!):
chain = prompt | (guardrails | llm_with_tools)
```

The `RunnableRails` object intercepts messages flowing to the LLM, applies input rails, then applies output rails to the response — all transparently within the LangChain pipeline.

### ⚠️ Important: `passthrough=True`
When `passthrough=True`, messages that pass the guardrails check are forwarded to the LLM unchanged. When `False`, guardrails may modify or rewrite the messages before forwarding.

In [ ]:
# 📚 Import all required components for the guarded agent
from langgraph.prebuilt import ToolNode, tools_condition  # LangGraph tool execution + routing
from nemoguardrails import RailsConfig                     # NeMo config loader
import os                                                  # OS utilities
from typing import Annotated                               # Type annotation support
from langchain_core.messages import BaseMessage            # Base message type
from langchain_core.prompts import ChatPromptTemplate      # Prompt template builder
from langchain_openai import ChatOpenAI                    # OpenAI chat model (for reference)
from langgraph.graph import StateGraph, START              # Graph builder + START node
from langgraph.graph.message import add_messages           # Message accumulator for state
from typing_extensions import TypedDict                    # Typed dictionary for state schema
from nemoguardrails.integrations.langchain.runnable_rails import RunnableRails  # 🛡️ LangChain guardrails wrapper

# 📁 Path to our guardrails configuration directory
config_path = "configs"

# 📊 Define the agent's state schema
# This TypedDict defines what data flows through the LangGraph nodes
class State(TypedDict):
    messages: Annotated[list, add_messages]             # Message history with automatic accumulation
    # add_messages ensures new messages are APPENDED (not replaced) in the state

# 🏗️ Main function to build the guarded tool-calling agent
async def create_tool_calling_agent():
    """
    Creates a LangGraph agent with NeMo Guardrails protection.
    
    Architecture:
        User → [Guardrails Input Check] → LLM (with tools) → [Tool Execution] → LLM → [Guardrails Output Check] → Response
    
    Returns:
        CompiledGraph: A ready-to-use LangGraph agent with guardrails
    """
    # 🧠 Use our previously configured Azure OpenAI LLM
    llms = llm
    
    # 🔧 Define the tools available to this agent
    tools = [search_knowledge, multiply]               # Knowledge search + math multiplication
    
    # 🔗 Bind tools to the LLM — this enables the LLM to generate tool calls
    llm_with_tools = llms.bind_tools(tools)
    
    # 🛡️ Initialize NeMo Guardrails from our config directory
    config = RailsConfig.from_path(config_path)        # Load config.yaml, prompt.yaml, rail.co
    guardrails = RunnableRails(
        config=config,                                 # The guardrails configuration
        passthrough=True,                              # Pass safe messages through unchanged
        verbose=True                                   # Enable detailed logging for debugging
    )
    
    # 📝 Define the system prompt — tells the LLM its role and capabilities
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant with access to tools."),
        ("placeholder", "{messages}"),                 # {messages} will be filled with conversation history
    ])
    
    # ⚡ Create the guarded runnable chain:
    # prompt → guardrails → llm_with_tools
    # The guardrails sit BETWEEN the prompt and the LLM, intercepting messages
    runnable_with_guardrails = prompt | (guardrails | llm_with_tools)
    
    # 🤖 Define the chatbot node function — this is what runs at the "chatbot" node
    async def chatbot(state: State):
        """Process user messages through guardrails and the LLM."""
        result = await runnable_with_guardrails.ainvoke(state)  # Run the guarded chain
        return {"messages": [result]}                  # Return the LLM response as a new message
    
    # 🏗️ Build the LangGraph StateGraph
    graph_builder = StateGraph(State)                  # Create graph with our State schema
    
    # 📌 Add the chatbot node — processes messages through guardrails + LLM
    graph_builder.add_node("chatbot", chatbot)
    
    # 🔧 Add the tool execution node — runs tool calls made by the LLM
    tool_node = ToolNode(tools=tools)                  # Wraps our tools for automatic execution
    graph_builder.add_node("tools", tool_node)
    
    # 🔀 Add conditional routing — if LLM requests a tool, route to "tools" node
    # If no tool is needed, the agent finishes (routes to END)
    graph_builder.add_conditional_edges(
        "chatbot",                                     # Source node
        tools_condition,                               # Built-in condition that checks for tool calls
    )
    
    # 🔄 After tool execution, route back to chatbot for the LLM to process results
    graph_builder.add_edge("tools", "chatbot")
    
    # 🚀 Set the entry point — conversations start at the chatbot node
    graph_builder.add_edge(START, "chatbot")
    
    # ✅ Compile and return the graph — ready for use!
    return graph_builder.compile()

# 🚀 Create the guarded agent
graph = await create_tool_calling_agent()

# 🧪 TEST: Ask "what is 2 * 2" — this should:
#   1. Pass input guardrails (safe math question)
#   2. LLM decides to call the multiply tool
#   3. Tool returns 4
#   4. LLM formats the response
#   5. Pass output guardrails (safe math answer)
result = await graph.ainvoke({
    "messages": [{"role": "user", "content": "what is 2 *2"}]
})

# 📊 Display the result
print("🎉 Agent response with guardrails:")
for msg in result["messages"]:
    print(f"  {msg.type}: {msg.content}")

## 8. 🎓 Summary & Key Takeaways

### Congratulations! You've Built a Safe AI Agent! 🏆

In this notebook, you learned how to add **programmable safety guardrails** to LLM applications using NVIDIA NeMo Guardrails.

### 📋 What We Covered:

| Section | Topic | Key Concept |
|---------|-------|-------------|
| **1** | Environment Setup | Installing NeMo Guardrails package |
| **2** | Azure OpenAI Config | Setting up the LLM backend |
| **3** | Config-Based Rails | Loading safety rules from YAML + Colang files |
| **4** | Colang Language | Writing human-readable guardrail rules |
| **5** | Custom Actions | Registering Python functions as Colang actions |
| **6** | Tool Definitions | Creating LangChain tools for agent capabilities |
| **7** | Agent Integration | Combining guardrails with LangGraph tool-calling agents |

### 🔑 Key Concepts to Remember:

1. **Three Types of Rails**: Input rails (screen user messages), output rails (validate LLM responses), and hallucination rails (check factual grounding)
2. **Colang Language**: A simple, readable way to define conversational safety rules with `define user`, `define bot`, and `define flow`
3. **Action Registration**: Python functions must be explicitly registered with `rails.register_action()` before Colang can call them
4. **RunnableRails**: NeMo's LangChain integration that wraps any Runnable with guardrails using the pipe (`|`) operator
5. **Config Directory**: Guardrails rules are stored in a `configs/` directory with `config.yaml`, `prompt.yaml`, and `rail.co` files

### 🚀 Next Steps:
- **Experiment**: Try sending political messages to the Colang-based rails and observe the deflection
- **Customize**: Write your own Colang rules for different topics (e.g., medical advice, financial guidance)
- **Extend**: Add more tools to the agent and test how guardrails protect them
- **Production**: Explore NeMo Guardrails' advanced features like knowledge-base grounding, topical rails, and multi-turn dialog management

### 🛡️ Remember:
> *"Guardrails are not limitations — they are the foundation of trustworthy AI. Every production AI system should have safety rails that protect users, organizations, and society."*

---

*📚 For more information, visit the [NeMo Guardrails Documentation](https://docs.nvidia.com/nemo/guardrails/) and the [GitHub Repository](https://github.com/NVIDIA/NeMo-Guardrails).*